In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem.SaltRemover import SaltRemover
from chemprop import data, featurizers, models, nn
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
import numpy as np 
import torch

In [2]:
df = pd.read_csv("../data/chembl_data.csv")

In [3]:
def remove_salts(smiles):
    remover = SaltRemover()
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    stripped_mol = remover.StripMol(mol)
    return Chem.MolToSmiles(stripped_mol)

In [4]:
df["smiles"] = df["smiles"].apply(lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s)))
df = df.dropna(subset=["smiles"])
df["smiles"] = df["smiles"].apply(remove_salts)
df = df.dropna(subset=["smiles"])
df = df[~df["smiles"].str.contains(r"\.")]
df = df.groupby("smiles")["pIC50"].median().reset_index()

In [5]:
df.head()

,smiles,pIC50
0,C#CCCC1(CCN2CCCc3ccc(NS(=O)(=O)c4ccc(-c5ccccc5...,5.32
1,C#CCNS(=O)(=O)c1ccc(CCN(CCc2cc(Cl)ccc2OCCC)C(=...,6.23
2,C/C(=N\NS(=O)(=O)c1ccc(C)cc1)c1nc2ccccc2o1,4.86
3,C/C1=C\CC[C@]2(C)O[C@@H]2[C@H]2OC(=O)[C@@H](CN...,5.66
4,C/C1=C\CC[C@]2(C)O[C@@H]2[C@H]2OC(=O)[C@@H](CN...,6.00


In [6]:
data_md = [data.MoleculeDatapoint.from_smi(smi, [target]) for smi, target in zip(df["smiles"], df["pIC50"])]

In [7]:
mols = [d.mol for d in data_md]
train_indices, val_indices, test_indices = data.make_split_indices(mols, split = 'scaffold_balanced', sizes = (0.8, 0.1, 0.1), seed = 42)

The return type of make_split_indices has changed in v2.1 - see help(make_split_indices)
/opt/miniconda3/envs/nlrp3/lib/python3.11/site-packages/astartes/samplers/extrapolation/scaffold.py:48: NoMatchingScaffold: No matching scaffold was found for the 1 molecules corresponding to indices {393}
  warnings.warn(


In [8]:
train_data, val_data, test_data = data.split_data_by_indices(data_md, train_indices, val_indices, test_indices)

In [9]:
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
train_dset = data.MoleculeDataset(train_data[0], featurizer)
val_dset = data.MoleculeDataset(val_data[0], featurizer)
test_dset = data.MoleculeDataset(test_data[0], featurizer)

In [10]:
train_loader = data.build_dataloader(train_dset)
val_loader = data.build_dataloader(val_dset, shuffle = False)
test_loader = data.build_dataloader(test_dset, shuffle = False)

Dropping last batch of size 1 to avoid issues with batch normalization     (dataset size = 65, batch_size = 64)
Dropping last batch of size 1 to avoid issues with batch normalization     (dataset size = 65, batch_size = 64)


In [12]:
def compute_metrics(model, loader):
    model.eval()
    preds = []
    actuals = []
    
    with torch.no_grad():
        for batch in loader:
            bmg, V_d, X_d, targets, *_ = batch
            pred = model(bmg, V_d, X_d)
            preds.append(pred.numpy())
            actuals.append(targets.numpy())
    
    preds = np.concatenate(preds).flatten()
    actuals = np.concatenate(actuals).flatten()
    rmse = np.sqrt(np.mean((preds - actuals) ** 2))
    ss_res = np.sum((actuals - preds) ** 2)
    ss_tot = np.sum((actuals - actuals.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot
    
    return (rmse, r2)

### Tuning

In [57]:
depths = [1, 2, 3, 4]
dropouts = [0.0, 0.2, 0.4]
hidden_sizes = [64, 128, 300]
metric_list = [nn.metrics.RMSE(), nn.metrics.R2Score()]

results = []

for depth in depths:
    for dropout in dropouts:
        for hidden in hidden_sizes:
            mp = nn.BondMessagePassing(depth = depth, d_h = hidden)
            agg = nn.MeanAggregation()
            ffn = nn.RegressionFFN(input_dim = hidden, dropout = dropout, hidden_dim = hidden)
            mpnn = models.MPNN(mp, agg, ffn, metric_list)

            trainer = pl.Trainer(
                logger = False,
                enable_checkpointing = False,
                enable_progress_bar = True,
                accelerator = "auto",
                devices = 1,
                max_epochs = 50,
            )
            
            trainer.fit(mpnn, train_loader, val_loader)

            val_metrics = compute_metrics(mpnn, val_loader)
            results.append({
                'depth': depth,
                'dropout': dropout,
                'hidden': hidden,
                'val_rmse': val_metrics[0],
                'val_r2': val_metrics[1]
            })
            print(f"depth={depth} dropout={dropout} hidden={hidden} val_rmse={val_metrics[0]:.3f} val_r2={val_metrics[1]:.3f}")

best = min(results, key=lambda r: r['val_rmse'])
print(f"\nBest config: {best}")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Loading `train_dataloader` to estimate number of stepping batches.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing │ 18.4 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation    │      0 │ train │     0 │
│ 2 │ bn              │ BatchNorm1d        │    128 │ train │     0 │
│ 3 │ predictor       │ RegressionFFN      │  4.2 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity           │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList         │      0 │ train │     0 │
└───┴─────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 22.7 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 22.7 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 24                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=50` reached.


...

┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation    │      0 │ train │     0 │
│ 2 │ bn              │ BatchNorm1d        │    600 │ train │     0 │
│ 3 │ predictor       │ RegressionFFN      │ 90.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity           │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList         │      0 │ train │     0 │
└───┴─────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 318 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 318 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 24                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=50` reached.


depth=4 dropout=0.4 hidden=300 val_rmse=0.957 val_r2=0.317

Best config: {'depth': 4, 'dropout': 0.0, 'hidden': 128, 'val_rmse': np.float32(0.74718136), 'val_r2': np.float32(0.58357215)}


### Testing

In [13]:
metric_list = [nn.metrics.RMSE(), nn.metrics.R2Score()]
mp = nn.BondMessagePassing(depth = 4, d_h = 128)
agg = nn.MeanAggregation()
ffn = nn.RegressionFFN(input_dim = 128, dropout = 0, hidden_dim = 128)
mpnn = models.MPNN(mp, agg, ffn, metric_list)

trainer = pl.Trainer(
    logger = False,
    enable_checkpointing = False,
    enable_progress_bar = True,
    accelerator = "auto",
    devices = 1,
    max_epochs = 50,
)

trainer.fit(mpnn, train_loader, val_loader)

test_metrics = compute_metrics(mpnn, test_loader)
print(f"test_rmse={test_metrics[0]:.3f} test_r2={test_metrics[1]:.3f}")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Loading `train_dataloader` to estimate number of stepping batches.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing │ 53.1 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation    │      0 │ train │     0 │
│ 2 │ bn              │ BatchNorm1d        │    256 │ train │     0 │
│ 3 │ predictor       │ RegressionFFN      │ 16.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity           │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList         │      0 │ train │     0 │
└───┴─────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 70.0 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 70.0 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 24                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=50` reached.


test_rmse=1.065 test_r2=0.334


In [14]:
import numpy as np
import lightning.pytorch as pl

test_rmses, test_r2s = [], []

for seed in [42, 0, 1, 2, 3]:
    pl.seed_everything(seed)
    
    mp  = nn.BondMessagePassing(depth = 4, d_h = 128)
    agg = nn.MeanAggregation()
    ffn = nn.RegressionFFN(input_dim = 128, dropout = 0, hidden_dim = 128)
    mpnn = models.MPNN(mp, agg, ffn, metric_list)
    
    trainer = pl.Trainer(
        logger=False,
        enable_checkpointing=False,
        enable_progress_bar=True,
        accelerator="auto",
        devices=1,
        max_epochs=50,
    )
    trainer.fit(mpnn, train_loader, val_loader)
    
    m = compute_metrics(mpnn, test_loader)
    test_rmses.append(m[0])
    test_r2s.append(m[1])
    print(f"seed {seed}: rmse={m[0]:.3f} r2={m[1]:.3f}")

print(f"\nTest RMSE: {np.mean(test_rmses):.3f} ± {np.std(test_rmses):.3f}")
print(f"Test R²:   {np.mean(test_r2s):.3f} ± {np.std(test_r2s):.3f}")

Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Loading `train_dataloader` to estimate number of stepping batches.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing │ 53.1 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation    │      0 │ train │     0 │
│ 2 │ bn              │ BatchNorm1d        │    256 │ train │     0 │
│ 3 │ predictor       │ RegressionFFN      │ 16.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity           │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList         │      0 │ train │     0 │
└───┴─────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 70.0 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 70.0 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 24                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=50` reached.
Seed set to 0
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Loading `train_dataloader` to estimate number of stepping batches.


seed 42: rmse=0.938 r2=0.484


...

seed 2: rmse=0.890 r2=0.535


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing │ 53.1 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation    │      0 │ train │     0 │
│ 2 │ bn              │ BatchNorm1d        │    256 │ train │     0 │
│ 3 │ predictor       │ RegressionFFN      │ 16.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity           │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList         │      0 │ train │     0 │
└───┴─────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 70.0 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 70.0 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 24                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=50` reached.


seed 3: rmse=1.023 r2=0.385

Test RMSE: 1.005 ± 0.083
Test R²:   0.403 ± 0.098


In [15]:
mpnn.eval()
test_preds = []
test_actuals = []

with torch.no_grad():
    for batch in test_loader:
        bmg, V_d, X_d, targets, *_ = batch
        pred = mpnn(bmg, V_d, X_d)
        test_preds.append(pred.numpy())
        test_actuals.append(targets.numpy())

test_preds = np.concatenate(test_preds).flatten()
test_actuals = np.concatenate(test_actuals).flatten()

In [16]:
def enrichment_factor(y_true, scores, fraction=0.01):
    n = len(y_true)
    n_top = max(1, int(n * fraction))
    
    top_idx = np.argsort(scores)[::-1][:n_top]
    
    actives_in_top = y_true[top_idx].sum()
    total_actives = y_true.sum()
    
    ef = (actives_in_top / total_actives) / fraction
    return ef

y_true = (test_actuals >= 6.0).astype(int)
scores = test_preds

print(f"EF1%:  {enrichment_factor(y_true, scores, 0.01):.2f}")
print(f"EF5%:  {enrichment_factor(y_true, scores, 0.05):.2f}")
print(f"EF10%: {enrichment_factor(y_true, scores, 0.10):.2f}")

EF1%:  3.23
EF5%:  1.94
EF10%: 1.94
